# VisionGuard: nine-class YOLO fine-tuning

Start from pretrained **YOLOv8n** and fine-tune for stop, yield, no entry, speed limits 40/60/80 km/h, and red/yellow/green traffic lights.

**You need a compact YOLO detection dataset ZIP before running this notebook.** Aim initially for roughly 2,000–5,000 varied images covering all nine classes. This notebook does not fetch the large Mapillary, Bosch, TT100K or BDD100K archives.

Export exactly the selected classes, with `data.yaml` and `train/images`, `train/labels`, `valid/images`, `valid/labels`, `test/images`, `test/labels`. Every image needs a label file, which is empty for negative images. Use bounding boxes in full road scenes, rather than classification-only sign crops. Label all selected signs and light states visible in every image. A generic traffic-light label is insufficient for learning its state. Speed-limit labels must be in km/h.

Keep different recording sequences/journeys in different splits and remove near-duplicates. Exact duplicate images across splits are checked automatically. Include each class in every split. Do not merge sign and light datasets without labelling any missing selected objects in those images.

In Colab choose **Runtime → Change runtime type → GPU**, then run cells in order. Free GPU availability varies. Training has not been run yet; metrics appear only after actual training.

References: [Ultralytics training](https://docs.ultralytics.com/modes/train/), [YOLO dataset format](https://docs.ultralytics.com/datasets/detect/), [Colab](https://docs.ultralytics.com/integrations/google-colab/).


In [ ]:
%pip install -q "ultralytics>=8.3,<9" pyyaml
import torch
assert torch.cuda.is_available(), "Enable a GPU runtime before training."
print(torch.cuda.get_device_name(0))


## Get only the compact dataset
Paste a direct HTTPS ZIP download link when prompted, or leave it blank to upload your compact ZIP to Colab. URLs are entered privately and are not stored in the notebook. The download limit is 1 GiB, and extraction is limited to 3 GiB. If an export is larger, create a smaller export first. The ZIP is downloaded to the cloud runtime, not to your computer.


In [ ]:
import getpass
import shutil
import stat
import urllib.request
import zipfile
from pathlib import Path
from urllib.parse import urlparse
from google.colab import files

workspace = Path('/content/visionguard')
workspace.mkdir(exist_ok=True)
archive = workspace / 'compact_dataset.zip'
MAX_DOWNLOAD = 1024 ** 3
url = getpass.getpass('Direct dataset ZIP URL (blank to upload): ').strip()
if url:
    assert urlparse(url).scheme == 'https', 'Use an HTTPS download link.'
    request = urllib.request.Request(url, headers={'User-Agent': 'VisionGuard-training'})
    with urllib.request.urlopen(request, timeout=60) as response:
        assert urlparse(response.url).scheme == 'https', 'Redirect must also use HTTPS.'
        length = int(response.headers.get('Content-Length', 0))
        assert length <= MAX_DOWNLOAD, 'Dataset exceeds the 1 GiB limit.'
        received = 0
        with archive.open('wb') as output:
            while chunk := response.read(1024 ** 2):
                received += len(chunk)
                assert received <= MAX_DOWNLOAD, 'Dataset exceeds the 1 GiB limit.'
                output.write(chunk)
else:
    uploaded = files.upload()
    assert len(uploaded) == 1, 'Upload exactly one dataset ZIP.'
    filename, content = next(iter(uploaded.items()))
    assert len(content) <= MAX_DOWNLOAD, 'Dataset exceeds the 1 GiB limit.'
    archive.write_bytes(content)
    del uploaded, content

dataset_dir = workspace / 'dataset'
assert not dataset_dir.exists(), 'Use a fresh runtime before importing a different dataset.'
with zipfile.ZipFile(archive) as bundle:
    assert sum(info.file_size for info in bundle.infolist()) <= 3 * 1024 ** 3, 'Export exceeds the extraction limit.'
    for info in bundle.infolist():
        destination = (dataset_dir / info.filename).resolve()
        assert destination.is_relative_to(dataset_dir.resolve()), 'Unsafe ZIP member path.'
        assert not stat.S_ISLNK(info.external_attr >> 16), 'ZIP symlinks are not supported.'
    bundle.extractall(dataset_dir)
yaml_files = list(dataset_dir.rglob('data.yaml'))
assert len(yaml_files) == 1, 'Dataset must contain exactly one data.yaml.'
source_yaml = yaml_files[0]
print('Dataset imported:', source_yaml)


In [ ]:
%%writefile /content/visionguard/train_limited.py
"""Validate a compact YOLO detection dataset and fine-tune pretrained YOLOv8."""
import argparse
import hashlib
import json
import math
from pathlib import Path

import yaml

CLASSES = [
    "stop sign", "yield sign", "no entry", "speed limit 40", "speed limit 60",
    "speed limit 80", "red traffic light", "yellow traffic light", "green traffic light",
]
ALIASES = {"stop": "stop sign", "yield": "yield sign", "give way": "yield sign",
           "red": "red traffic light", "yellow": "yellow traffic light",
           "amber": "yellow traffic light", "green": "green traffic light"}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def canonical_name(name):
    name = " ".join(str(name).lower().replace("_", " ").replace("-", " ").split())
    return ALIASES.get(name, name)


def prepare_data(source, output):
    source = Path(source).resolve()
    output = Path(output).resolve()
    root = source.parent
    config = yaml.safe_load(source.read_text(encoding="utf-8"))
    raw_names = config.get("names", [])
    if isinstance(raw_names, dict):
        raw_names = [raw_names.get(i, raw_names.get(str(i))) for i in range(len(raw_names))]
    names = [canonical_name(name) for name in raw_names]
    if len(names) != len(CLASSES) or set(names) != set(CLASSES):
        raise ValueError(f"Dataset must contain exactly these nine classes: {CLASSES}. "
                         f"Found {names}. Export a detection dataset with these classes first.")
    # Keep class IDs in the export's original order, while normalizing names.
    prepared = {"names": names, "nc": len(names)}
    counts = {}
    seen = {}
    for split in ("train", "val", "test"):
        value = config.get(split)
        if not isinstance(value, str):
            raise ValueError(f"Provide a {split} image directory in data.yaml.")
        split_alias = "valid" if split == "val" else split
        candidates = [root / value, root / str(config.get("path", ".")) / value,
                      root / split_alias / "images", root / split / "images"]
        folder = next((p.resolve() for p in candidates if p.is_dir()
                       and p.resolve().is_relative_to(root)), None)
        if folder is None:
            raise ValueError(f"Cannot find {split} images inside {root}.")
        if folder.name != "images":
            raise ValueError("Use train/images, valid/images and test/images with sibling labels folders.")
        images = sorted(p for p in folder.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)
        if not images:
            raise ValueError(f"Empty {split} split.")
        split_counts = [0] * len(names)
        for image in images:
            digest = hashlib.sha256(image.read_bytes()).hexdigest()
            previous = seen.get(digest)
            if previous is not None and previous != split:
                raise ValueError(f"Identical image appears in {previous} and {split}: {image.name}")
            seen[digest] = split
            label = folder.parent / "labels" / image.relative_to(folder).with_suffix(".txt")
            if not label.is_file():
                raise ValueError(f"Missing label {label}; negative images need an empty label file.")
            for line in label.read_text(encoding="utf-8").splitlines():
                if not line.strip():
                    continue
                parts = line.split()
                if len(parts) != 5:
                    raise ValueError(f"Expected detection label: class x y width height in {label}")
                class_id = int(parts[0])
                box = [float(v) for v in parts[1:]]
                if class_id not in range(len(names)) or not all(math.isfinite(v) and 0 <= v <= 1 for v in box):
                    raise ValueError(f"Invalid class ID or normalized coordinates in {label}")
                x, y, width, height = box
                if width <= 0 or height <= 0 or x-width/2 < -0.001 or x+width/2 > 1.001 or y-height/2 < -0.001 or y+height/2 > 1.001:
                    raise ValueError(f"Bounding box extends outside the image in {label}")
                split_counts[class_id] += 1
        missing = [names[i] for i, count in enumerate(split_counts) if count == 0]
        if missing:
            raise ValueError(f"{split} has no examples for: {missing}. Add examples before training.")
        prepared[split] = str(folder)
        counts[split] = {"images": len(images), "objects": dict(zip(names, split_counts))}
    output.mkdir(parents=True, exist_ok=True)
    data = output / "data.yaml"
    # No dataset download directive is copied from the source export.
    data.write_text(yaml.safe_dump(prepared, sort_keys=False), encoding="utf-8")
    (output / "dataset_counts.json").write_text(json.dumps(counts, indent=2), encoding="utf-8")
    print(json.dumps(counts, indent=2))
    return data


def train(data, output, epochs=50, imgsz=960):
    from ultralytics import YOLO
    existing_weights = Path(__file__).resolve().parents[1] / "webapp" / "yolov8n.pt"
    model = YOLO(str(existing_weights) if existing_weights.is_file() else "yolov8n.pt")
    model.train(data=str(data), epochs=epochs, imgsz=imgsz, batch=8, patience=12,
                device=0, workers=2, seed=42, deterministic=True, cache=False,
                fliplr=0.0, flipud=0.0, hsv_h=0.0, hsv_s=0.0,
                project=str(output), name="road_signs_lights", exist_ok=False)
    best = Path(model.trainer.best)
    evaluation = YOLO(str(best)).val(data=str(data), split="test", imgsz=imgsz,
                                   device=0, project=str(output), name="held_out_test")
    metrics = {"mAP50": float(evaluation.box.map50), "mAP50_95": float(evaluation.box.map),
               "precision": float(evaluation.box.mp), "recall": float(evaluation.box.mr)}
    (best.parent.parent / "test_metrics.json").write_text(json.dumps(metrics, indent=2))
    print(f"Trained checkpoint: {best}")
    return best


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data", required=True, help="Compact nine-class YOLO export's data.yaml")
    parser.add_argument("--output", default="training_runs")
    parser.add_argument("--epochs", type=int, default=50)
    parser.add_argument("--imgsz", type=int, default=960)
    parser.add_argument("--validate-only", action="store_true")
    args = parser.parse_args()
    data = prepare_data(args.data, Path(args.output) / "prepared")
    if not args.validate_only:
        train(data, args.output, args.epochs, args.imgsz)


## Validate class names and annotations
Accepted aliases include `stop`, `yield`, `give way`, `red`, `yellow`, `amber`, `green`, and underscore/hyphen variants of the canonical names. Class IDs retain the export's order. Other class names must be renamed explicitly in the export; do not assign a generic light label to a colour.


In [ ]:
import sys
sys.path.insert(0, str(workspace))
from train_limited import prepare_data, train
prepared_yaml = prepare_data(source_yaml, workspace / 'prepared')


## Fine-tune and evaluate
Initial settings: 50 epochs, 960-pixel inputs, batch size 8, early stopping, and no flips or hue/saturation augmentation. These are starting settings, not an accuracy guarantee. If GPU memory is insufficient, reduce `batch=8` in the helper cell above, rerun it, and restart the Python runtime before importing it again. The held-out test split is evaluated once after training; use validation, not test results, to choose settings.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from datetime import datetime
run_dir = Path('/content/drive/MyDrive/VisionGuardTraining') / datetime.now().strftime('%Y%m%d_%H%M%S')
run_dir.mkdir(parents=True, exist_ok=False)
shutil.copy(workspace / 'prepared/dataset_counts.json', run_dir / 'dataset_counts.json')
best = train(prepared_yaml, run_dir, epochs=50, imgsz=960)


## Download the checkpoint
The trained checkpoint and training results are saved in your Google Drive. Download the checkpoint below and put it at `webapp/models/road_signs_lights.pt` in your local VisionGuard project, then restart the app. Until a checkpoint is installed, the app uses its pretrained fallback for stop signs and generic traffic lights only. For interrupted runs, retain `last.pt`; the dataset needs to be loaded again into a new runtime before resuming.


In [ ]:
checkpoint = workspace / 'road_signs_lights.pt'
shutil.copy(best, checkpoint)
files.download(str(checkpoint))
print('Install at: webapp/models/road_signs_lights.pt')
print('Test metrics:', best.parent.parent / 'test_metrics.json')
